# ESG × Climate Attention Shock — Panel Regression with Interaction

**Objectif.** Tester si le greenium s'amplifie lors des **chocs d'attention médiatique au changement climatique**, en suivant le mécanisme théorique de Pástor, Stambaugh & Taylor (*JFE*, 2022) et l'implémentation empirique d'Ardia, Bluteau, Boudt & Inghelbrecht (*Management Science*, 2023).

---

## Logique économique

Selon Pástor-Stambaugh-Taylor :
- À l'**équilibre**, les firmes vertes ont un rendement *attendu* plus faible (greenium structurel).
- Mais **ex-post**, lors d'un choc positif d'attention au climat, les investisseurs réallouent vers les firmes vertes, leurs prix montent, donc leurs rendements *réalisés* sont temporairement plus élevés.
- L'effet attendu est donc : un **rendement réalisé plus élevé pour les firmes ESG lors des chocs MCCC positifs**.

C'est exactement ce qu'on teste avec un terme d'interaction $ESG 	imes 	ext{Shock}^{MCCC}$ :

$$R^{ex}_{i,t} = \alpha + \beta_1 ESG_{i,t} + \beta_2 \text{Shock}^{MCCC}_t + \beta_3 (ESG_{i,t} \times \text{Shock}^{MCCC}_t) + \text{controls} + \varepsilon_{i,t}$$

**Interprétation des coefficients :**
- $\beta_1 < 0$ : greenium structurel (effet ESG en l'absence de choc, $\text{Shock}=0$)
- $\beta_2$ : effet du choc sur la firme moyenne (typiquement non significatif, capté par les FE de date)
- $\beta_3 > 0$ : **les firmes à haut ESG bénéficient davantage des chocs positifs d'attention climat** — confirmation empirique de PST

---

## Construction du choc MCCC

Le MCCC brut a une auto-corrélation très élevée (~0.85) — c'est une série persistante. Pour isoler la composante **inattendue** (qui est ce qui devrait bouger les prix selon la théorie d'efficience), on extrait le **résidu d'un AR(1)** :

$$\text{MCCC}_t = c + \rho \cdot \text{MCCC}_{t-1} + \text{Shock}^{MCCC}_t$$

Le shock ainsi construit a une moyenne nulle, capture la part imprévisible, et peut être interprété comme une "surprise" mensuelle d'attention climat.


In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.arima.model import ARIMA
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 10


## 1. Chargement des données

In [ ]:
# === ADAPT THESE PATHS ===
ESG_DATA_PATH    = "C:/Users/anton/OneDrive/Bureau/Edhec course/Master 2/Research Thesis/Data-Analysis/Data/price_esg/ESG_Panel_With_Closing_Price.xlsx"
FF_FACTORS_PATH  = "C:/Users/anton/OneDrive/Bureau/Edhec course/Master 2/Research Thesis/Data-Analysis/Data/fama_french_factors/5_factors ff_US.csv"
MCAP_PATH        = "C:/Users/anton/OneDrive/Bureau/Edhec course/Master 2/Research Thesis/Data-Analysis/Data/SP500_Mcap_Panel.xlsx"
ACCOUNTING_PATH  = "C:/Users/anton/OneDrive/Bureau/Edhec course/Master 2/Research Thesis/Data-Analysis/Data/SP500_Panel_Data_Fama_Factors.xlsx"
MCCC_PATH        = "C:/Users/anton/OneDrive/Bureau/Edhec course/Master 2/Research Thesis/Data-Analysis/Data/Sentometrics_US_Media_Climate_Change_Index.xlsx"

# Estimation parameters
BETA_WINDOW   = 36
BETA_MIN_OBS  = 24
NW_LAGS       = 12  # Increased to 12 because annual accounting variables create up to 12-month autocorrelation


In [ ]:
# Load ESG panel and build returns (same as previous notebooks)
data = pd.read_excel(ESG_DATA_PATH, sheet_name="Panel Data")
data["Date"]          = pd.to_datetime(data["Date"]).values.astype("datetime64[M]")
data["Closing_Price"] = pd.to_numeric(data["Closing_Price"], errors="coerce")
data = data.dropna(subset=["Closing_Price", "ESG"]).copy()
data = data.sort_values(["Ticker", "Date"])
data["Return"] = data.groupby("Ticker")["Closing_Price"].pct_change()

# Fama-French factors
ff = pd.read_csv(FF_FACTORS_PATH, skiprows=4)
ff.columns = ['Date', 'Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'RF']
ff['Date'] = ff['Date'].astype(str).str.strip()
ff = ff[ff['Date'].str.len() == 6].copy()
ff['Date'] = pd.to_datetime(ff['Date'], format='%Y%m').values.astype("datetime64[M]")
for c in ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'RF']:
    ff[c] = pd.to_numeric(ff[c], errors='coerce') / 100.0

data = data.merge(ff[['Date', 'RF', 'Mkt-RF']], on='Date', how='left')
data['ExRet'] = data['Return'] - data['RF']
print(f"Panel: {data.shape[0]:,} obs | {data['Ticker'].nunique()} tickers | "
      f"{data['Date'].min().date()} → {data['Date'].max().date()}")


## 2. Chargement du MCCC et construction du choc

On utilise la **version 2025 mensuelle** du MCCC (Sentometrics), couvrant 2003-01 à 2025-06. La colonne `Aggregate` est l'indice composite principal (Ardia et al. 2023).

**Construction du shock.** On régresse $\text{MCCC}_t$ sur sa propre valeur retardée et on garde le résidu :

$$\text{MCCC}_t = c + \rho \cdot \text{MCCC}_{t-1} + \text{Shock}^{MCCC}_t$$

Le choc est ensuite **standardisé** (moyenne 0, écart-type 1) pour faciliter l'interprétation des coefficients d'interaction.


In [ ]:
# Load MCCC 2025 monthly (header is on row 6)
mccc = pd.read_excel(MCCC_PATH, sheet_name='2025 update monthly', header=6)
mccc['Date'] = pd.to_datetime(mccc['Date']).values.astype("datetime64[M]")
mccc = mccc[['Date', 'Aggregate']].rename(columns={'Aggregate': 'MCCC'}).sort_values('Date').reset_index(drop=True)

print(f"MCCC: {len(mccc)} months | {mccc['Date'].min().date()} → {mccc['Date'].max().date()}")
print(f"\nMCCC stats:")
print(mccc['MCCC'].describe().round(3))
print(f"\nAR(1) of raw MCCC: {mccc['MCCC'].autocorr(1):.3f}  ← high persistence, need to extract shocks")


In [ ]:
# Fit AR(1) on MCCC and extract residuals as the climate attention shock
ar1 = ARIMA(mccc['MCCC'].dropna(), order=(1, 0, 0)).fit()
print(ar1.summary().tables[1])

# Residuals = the unpredictable component of MCCC
mccc['Shock_raw'] = ar1.resid.values
# Standardize for easier interpretation (coef = effect of a 1-σ shock)
mccc['Shock'] = (mccc['Shock_raw'] - mccc['Shock_raw'].mean()) / mccc['Shock_raw'].std()

print(f"\nShock stats (standardized):")
print(mccc['Shock'].describe().round(3))
print(f"AR(1) of shock: {mccc['Shock'].autocorr(1):.3f}  ← should be near 0 by construction")


In [ ]:
# Plot MCCC level and standardized shocks
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

ax = axes[0]
ax.plot(mccc['Date'], mccc['MCCC'], color='#2E86AB', linewidth=1.5)
ax.set_ylabel('MCCC level (Aggregate)')
ax.set_title('Media Climate Change Concerns — Aggregate index, monthly (Ardia et al., 2023)')
ax.grid(True, alpha=0.3)

ax = axes[1]
colors = np.where(mccc['Shock'] >= 0, '#2E86AB', '#C73E1D')
ax.bar(mccc['Date'], mccc['Shock'], width=25, color=colors, alpha=0.7)
ax.axhline(0, color='black', linestyle='--', alpha=0.6)
ax.axhline( 1.96, color='gray', linestyle=':', alpha=0.5, label='±1.96σ')
ax.axhline(-1.96, color='gray', linestyle=':', alpha=0.5)
ax.set_ylabel('Shock (standardized AR(1) residual)')
ax.set_xlabel('Date')
ax.set_title('Climate attention SHOCK = AR(1) residual of MCCC, standardized')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('MCCC_shocks.png', dpi=150, bbox_inches='tight')
plt.show()


## 3. Ajout des contrôles firme (bêta marché, size, BM, OP, INV)

In [ ]:
# Rolling market beta (same as in Fama-MacBeth notebook)
def rolling_beta(group, window=BETA_WINDOW, min_obs=BETA_MIN_OBS):
    y = group['ExRet']; x = group['Mkt-RF']
    cov_xy = y.rolling(window, min_periods=min_obs).cov(x)
    var_x  = x.rolling(window, min_periods=min_obs).var()
    return (cov_xy / var_x).shift(1)

data = data.sort_values(['Ticker', 'Date'])
data['Beta_MKT'] = data.groupby('Ticker', group_keys=False).apply(rolling_beta)

# Market cap
mcap = pd.read_excel(MCAP_PATH)
mcap['Date'] = pd.to_datetime(mcap['Date']).values.astype("datetime64[M]")
mcap['Mcap (€M)'] = pd.to_numeric(mcap['Mcap (€M)'], errors='coerce')
mcap = mcap.dropna(subset=['Mcap (€M)'])
data = data.merge(mcap[['Ticker','Date','Mcap (€M)']], on=['Ticker','Date'], how='left')
data['log_Mcap'] = np.log(data['Mcap (€M)'])

# Accounting (annual, lagged 1Y)
acc = pd.read_excel(ACCOUNTING_PATH)
for c in ['Assets (€000)','Equity (€000)','EBIT (€000)']:
    acc[c] = pd.to_numeric(acc[c], errors='coerce')
acc = acc.rename(columns={'Assets (€000)':'Assets','Equity (€000)':'Equity','EBIT (€000)':'EBIT'})
acc = acc.sort_values(['Ticker','Year'])
acc['OP']  = acc['EBIT'] / acc['Equity'].replace(0, np.nan)
acc['INV'] = acc.groupby('Ticker')['Assets'].pct_change()
acc['MatchYear'] = acc['Year'] + 1
acc_lag = acc[['Ticker','MatchYear','Equity','OP','INV']].rename(columns={'MatchYear':'Year'})
data['Year'] = data['Date'].dt.year
data = data.merge(acc_lag, on=['Ticker','Year'], how='left')
data['BM'] = (data['Equity'] / 1000.0) / data['Mcap (€M)']

# Winsorize
def winsorize_xs(s, lo=0.01, hi=0.99):
    return s.clip(s.quantile(lo), s.quantile(hi))
for c in ['BM','OP','INV','log_Mcap','Beta_MKT']:
    data[c] = data.groupby('Date')[c].transform(winsorize_xs)

# Merge MCCC shock
data = data.merge(mccc[['Date','MCCC','Shock']], on='Date', how='left')
print(f"After all merges: {data.shape[0]:,} obs")
print(f"Coverage: {data[['ExRet','ESG','Shock','Beta_MKT','log_Mcap','BM','OP','INV']].notna().mean().round(3).to_dict()}")


## 4. Approche 1 — Two-step : régression du $\hat\gamma_{ESG,t}$ Fama-MacBeth sur le shock

Cette approche est la plus **propre méthodologiquement** : on utilise les coefficients Fama-MacBeth déjà estimés et on teste s'ils covarient avec le choc MCCC.

$$\hat\gamma_{ESG,t} = a + b \cdot \text{Shock}^{MCCC}_t + u_t$$

- $b > 0$ : un choc positif d'attention climat **augmente** le coefficient ESG (= rend le greenium moins négatif, voire positif) → cohérent avec PST (les firmes vertes performent mieux ex-post lors des chocs).
- $b < 0$ : effet inverse, peu cohérent avec la théorie.


In [ ]:
# Re-run Fama-MacBeth (full spec) to get the time series of gamma_ESG_t
def fama_macbeth(panel, y_col, x_cols, date_col='Date', min_obs=30):
    needed = [y_col] + x_cols
    df = panel.dropna(subset=needed).copy()
    coefs_rows = []
    for date, sub in df.groupby(date_col):
        if len(sub) < min_obs: continue
        X = sm.add_constant(sub[x_cols].values)
        y = sub[y_col].values
        res = sm.OLS(y, X).fit()
        coefs_rows.append({date_col: date,
                           **{n: c for n,c in zip(['const']+x_cols, res.params)}})
    return pd.DataFrame(coefs_rows).set_index(date_col).sort_index()

coefs = fama_macbeth(data, 'ExRet',
                     ['ESG', 'Beta_MKT', 'log_Mcap', 'BM', 'OP', 'INV'])
print(f"Estimated FM gammas for {len(coefs)} months")
print(f"\nMean γ_ESG (full sample) = {coefs['ESG'].mean():+.6f}")


In [ ]:
# Merge gamma_ESG_t with the MCCC shock
gamma_panel = coefs[['ESG']].rename(columns={'ESG': 'gamma_ESG'}).reset_index()
gamma_panel = gamma_panel.merge(mccc[['Date','Shock']], on='Date', how='inner')
gamma_panel = gamma_panel.dropna()
print(f"Joint sample: {len(gamma_panel)} months")

# Step-2 regression: gamma_ESG_t = a + b * Shock_t + u_t
X = sm.add_constant(gamma_panel['Shock'])
y = gamma_panel['gamma_ESG']
m2 = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': NW_LAGS})

print("="*78)
print("APPROACH 1 — Two-step: γ_ESG_t = a + b · Shock_MCCC_t  (Newey-West HAC, 12 lags)")
print("="*78)
print(m2.summary().tables[1])
print(f"\nR² : {m2.rsquared:.4f}")


In [ ]:
# Visual: scatter of gamma_ESG vs MCCC shock
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(gamma_panel['Shock'], gamma_panel['gamma_ESG'],
           color='#2E86AB', alpha=0.6, s=45, edgecolor='black', linewidth=0.5)

# Regression line
x_line = np.linspace(gamma_panel['Shock'].min(), gamma_panel['Shock'].max(), 100)
y_line = m2.params['const'] + m2.params['Shock'] * x_line
ax.plot(x_line, y_line, color='#C73E1D', linewidth=2.5,
        label=f"Slope = {m2.params['Shock']:+.6f}  (t = {m2.tvalues['Shock']:+.2f})")

ax.axhline(0, color='black', linestyle='--', alpha=0.5)
ax.axvline(0, color='black', linestyle='--', alpha=0.5)
ax.set_xlabel('MCCC shock (standardized)')
ax.set_ylabel('γ_ESG_t  (Fama-MacBeth coefficient)')
ax.set_title('Does the cross-sectional ESG coefficient react to climate attention shocks?')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('gamma_ESG_vs_shock.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. Approche 2 — Panel poolé avec interaction $ESG \times \text{Shock}$

Spécification panel directe :

$$R^{ex}_{i,t} = \alpha + \beta_1 ESG_{i,t} + \beta_2 \text{Shock}_t + \beta_3 (ESG_{i,t} \cdot \text{Shock}_t) + \beta^{T} X_{i,t} + \varepsilon_{i,t}$$

où $X_{i,t}$ inclut Beta_MKT, log_Mcap, BM, OP, INV.

**Standard errors :** double-clustering par firme et par date (Petersen 2009) pour gérer simultanément la corrélation cross-sectionnelle (chocs communs entre firmes à une date) et la persistance temporelle au sein d'une firme.

**Coefficient d'intérêt :** $\beta_3$ (l'interaction). Si $\beta_3 > 0$, ça veut dire que l'effet ESG sur le rendement est **plus positif** lors des chocs MCCC élevés → cohérent avec PST.


In [ ]:
# Build the panel with interaction term
panel = data.dropna(subset=['ExRet','ESG','Shock','Beta_MKT','log_Mcap','BM','OP','INV']).copy()
panel['ESG_x_Shock'] = panel['ESG'] * panel['Shock']

# Encode firm and date as integer codes for clustering
panel['firm_id'] = panel['Ticker'].astype('category').cat.codes
panel['date_id'] = panel['Date'].astype('category').cat.codes

print(f"Panel sample for interaction model: {len(panel):,} obs | "
      f"{panel['Ticker'].nunique()} firms | {panel['Date'].nunique()} months")


In [ ]:
# Estimate the panel regression with interaction, using two-way clustered SE
y = panel['ExRet'].values
X_cols = ['ESG', 'Shock', 'ESG_x_Shock', 'Beta_MKT', 'log_Mcap', 'BM', 'OP', 'INV']
X = sm.add_constant(panel[X_cols].values)

# Two-way cluster-robust SE (firm + date)
groups = np.column_stack([panel['firm_id'].values, panel['date_id'].values])
m_panel = sm.OLS(y, X).fit(cov_type='cluster', cov_kwds={'groups': groups})

# Pretty print
res_table = pd.DataFrame({
    'Coef'   : m_panel.params,
    'SE'     : m_panel.bse,
    't-stat' : m_panel.tvalues,
    'p-value': m_panel.pvalues,
}, index=['const'] + X_cols)

print("="*78)
print("APPROACH 2 — Panel pooled OLS with ESG × Shock interaction")
print("Two-way cluster-robust SE (firm + date)")
print("="*78)
print(res_table.round(6).to_string())
print(f"\nR² : {m_panel.rsquared:.4f}")
print(f"N  : {int(m_panel.nobs):,}")


## 6. Visualisation de l'effet conditionnel

L'effet marginal de l'ESG sur le rendement, **conditionnel au shock**, est :

$$\frac{\partial R^{ex}_{i,t}}{\partial ESG_{i,t}} = \beta_1 + \beta_3 \cdot \text{Shock}_t$$

Si $\beta_3 > 0$, la pente devient plus positive (ou moins négative) lors des chocs positifs d'attention climat. On peut visualiser cette pente conditionnelle pour différentes valeurs du shock.


In [ ]:
# Conditional marginal effect of ESG across shock values
b1 = m_panel.params[1]   # ESG
b3 = m_panel.params[3]   # ESG × Shock
se_b1 = m_panel.bse[1]
se_b3 = m_panel.bse[3]
cov_b1_b3 = m_panel.cov_params().iloc[1, 3]

# Range of shock values: cover the empirical distribution
shock_grid = np.linspace(panel['Shock'].quantile(0.02),
                         panel['Shock'].quantile(0.98), 100)
mfx       = b1 + b3 * shock_grid
mfx_se    = np.sqrt(se_b1**2 + (shock_grid**2) * se_b3**2 + 2 * shock_grid * cov_b1_b3)
mfx_lo    = mfx - 1.96 * mfx_se
mfx_hi    = mfx + 1.96 * mfx_se

# Annualize: monthly excess return per ESG point × 12
mfx_ann    = mfx * 12 * 100  # in %
mfx_lo_ann = mfx_lo * 12 * 100
mfx_hi_ann = mfx_hi * 12 * 100

fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(shock_grid, mfx_ann, color='#2E86AB', linewidth=2.5,
        label='Marginal effect of ESG (β₁ + β₃·Shock) × 12')
ax.fill_between(shock_grid, mfx_lo_ann, mfx_hi_ann, color='#2E86AB', alpha=0.2,
                label='95% confidence band')
ax.axhline(0, color='black', linestyle='--', alpha=0.6)
ax.axvline(0, color='gray', linestyle=':', alpha=0.6, label='Shock = 0 (no surprise)')

# Annotate effect at typical shock values
for s_val, lab in [(-1, 'Shock = -1σ'), (0, 'Shock = 0'), (+1, 'Shock = +1σ')]:
    eff = (b1 + b3 * s_val) * 12 * 100
    ax.scatter([s_val], [eff], s=80, color='#C73E1D', zorder=5, edgecolor='black')
    ax.annotate(f'{lab}\n{eff:+.3f}%/yr per ESG pt',
                xy=(s_val, eff), xytext=(s_val + 0.15, eff),
                fontsize=9, color='#C73E1D')

ax.set_xlabel('MCCC shock (standardized)')
ax.set_ylabel('Annualized return per +1 ESG point (%)')
ax.set_title('Marginal effect of ESG on returns conditional on climate attention shock')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('conditional_ESG_effect.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. Stabilité — analyse par sous-périodes des chocs

In [ ]:
# Split panel by tertiles of MCCC shock (negative shock / neutral / positive shock months)
panel['shock_terc'] = pd.qcut(panel['Shock'], 3, labels=['Low_shock','Mid_shock','High_shock'])

results_by_terc = []
for label in ['Low_shock','Mid_shock','High_shock']:
    sub = panel[panel['shock_terc'] == label]
    if len(sub) < 100: continue
    Xs = sm.add_constant(sub[['ESG','Beta_MKT','log_Mcap','BM','OP','INV']].values)
    ys = sub['ExRet'].values
    gs = np.column_stack([sub['firm_id'].values, sub['date_id'].values])
    m = sm.OLS(ys, Xs).fit(cov_type='cluster', cov_kwds={'groups': gs})
    results_by_terc.append({
        'Period':    label,
        'N obs':     len(sub),
        'Months':    sub['Date'].nunique(),
        'Avg shock': sub['Shock'].mean().round(3),
        'β_ESG':     f"{m.params[1]:+.6f}",
        'SE':        f"{m.bse[1]:.6f}",
        't-stat':    f"{m.tvalues[1]:+.2f}",
        'Annual %':  f"{m.params[1]*12*100:+.3f}%",
    })

print("="*90)
print("ESG coefficient by tercile of MCCC shock (panel OLS, 2-way cluster SE)")
print("="*90)
print(pd.DataFrame(results_by_terc).to_string(index=False))


In [ ]:
# Auto-summary of findings
print("="*78)
print("SUMMARY — ESG × Climate Attention Shock")
print("="*78)

# Two-step
b_step  = m2.params['Shock']
t_step  = m2.tvalues['Shock']
p_step  = m2.pvalues['Shock']

# Panel
b_int   = m_panel.params[3]
t_int   = m_panel.tvalues[3]
p_int   = m_panel.pvalues[3]
b_esg   = m_panel.params[1]
t_esg   = m_panel.tvalues[1]
p_esg   = m_panel.pvalues[1]

print(f"""
APPROACH 1 — Two-step on Fama-MacBeth γ_ESG_t:
  Coefficient of MCCC shock on γ_ESG_t: {b_step:+.6f}  (t = {t_step:+.2f}, p = {p_step:.3f})

APPROACH 2 — Panel with ESG × Shock interaction:
  Baseline ESG effect (β₁)    : {b_esg:+.6f}  (t = {t_esg:+.2f}, p = {p_esg:.3f})
  ESG × Shock interaction (β₃): {b_int:+.6f}  (t = {t_int:+.2f}, p = {p_int:.3f})

Interpretation:
  β₃ > 0 → high-ESG firms perform better (less greenium / more premium) during
            POSITIVE climate-attention shocks. This matches Pástor-Stambaugh-Taylor
            (2022) and Ardia et al. (2023): unexpected attention to climate drives
            green firms' realized returns up.

  β₃ < 0 → opposite pattern. High-ESG firms perform worse during positive shocks.
            Less consistent with PST theory.

  β₃ ≈ 0 → no time-varying effect; the greenium is structural, not driven by
            shocks of media attention.
""")


In [ ]:
# Export results
with pd.ExcelWriter('FM_ESG_x_MCCC_Results.xlsx', engine='openpyxl') as w:
    # Two-step
    pd.DataFrame({
        'Variable': ['const','Shock'],
        'Coef'    : [m2.params['const'], m2.params['Shock']],
        'SE'      : [m2.bse['const'],    m2.bse['Shock']],
        't'       : [m2.tvalues['const'],m2.tvalues['Shock']],
        'p'       : [m2.pvalues['const'],m2.pvalues['Shock']],
    }).to_excel(w, sheet_name='1_TwoStep_FM', index=False)
    # Panel
    res_table.to_excel(w, sheet_name='2_Panel_Interaction')
    # Tercile analysis
    pd.DataFrame(results_by_terc).to_excel(w, sheet_name='3_Tercile_Analysis', index=False)
    # Time series of FM gammas + shocks
    gamma_panel.to_excel(w, sheet_name='Gamma_Shock_TS', index=False)

print("✔ Saved FM_ESG_x_MCCC_Results.xlsx")
